In [59]:
import forte2
import numpy as np
import matplotlib.pyplot as plt
import itertools
import time

In [60]:
psi2=forte2.SparseState()
dets = forte2.hilbert_space(4,3,2)
for d in dets:
    c = np.random.rand()
    psi2.add(d,1-2*c)
psi2 *= 1/psi2.norm()
#psi2.norm()
psi2

|0a22000000000000000000000000000000000000000000000000000000000000> * (0.35135585,0.00000000)
|02a2000000000000000000000000000000000000000000000000000000000000> * (0.25556451,0.00000000)
|022a000000000000000000000000000000000000000000000000000000000000> * (0.11139428,0.00000000)
|baa2000000000000000000000000000000000000000000000000000000000000> * (0.01202030,0.00000000)
|ba2a000000000000000000000000000000000000000000000000000000000000> * (0.00573355,0.00000000)
|b2aa000000000000000000000000000000000000000000000000000000000000> * (-0.36625862,0.00000000)
|a022000000000000000000000000000000000000000000000000000000000000> * (0.12778311,0.00000000)
|aba2000000000000000000000000000000000000000000000000000000000000> * (-0.06634432,0.00000000)
|ab2a000000000000000000000000000000000000000000000000000000000000> * (-0.09922595,0.00000000)
|20a2000000000000000000000000000000000000000000000000000000000000> * (0.16690621,0.00000000)
|202a000000000000000000000000000000000000000000000000000000000000> 

In [61]:
op = forte2.SparseOperator() #analagous to wavefunction builder class, creates an empty operator
i = 1
j = 2
op.add(f"[{i}a+{j}b+]",1.0) #create an alpha electron in orb 1 and a beta electron in orb 2; 
#order matters in this function, can add allow_reordering=True
#takes formatted strings

#op.add([1,2],[3,4],[5,6],[7,8],2.0)# list: number of alpha creop, beta creop, alpha annop, beta annop
#order doesn't matter in this function, maybe not as consistent, could miss a sign

op

(1 + 0i) * [1a+ 2b+]

In [62]:
d = forte2.Determinant("0ab2") #many zeroes as stored placeholders; 0, alpha, beta, 2=doubly occupied
d1 = forte2.Determinant("20")
d2 = forte2.Determinant("02")
d1.str(2),d2.str(2) #show first two spin orbitals

c1=0.5
c2=-np.sqrt(1-c1**2)
psi = forte2.SparseState() #create a state using class 'sparse state'
psi.add(d1,c1) #be careful to only add once 
psi.add(d2,c2)

psi

phi = op @ psi

exp_val = psi.overlap(phi)
exp_val

0j

#### Code the 1-RDM using Forte2

In [63]:
#Create the wavefunction
wfxn = forte2.SparseState()
d1 = forte2.Determinant("aa00")
d2 = forte2.Determinant("00aa")
c1 = 1/np.sqrt(2)
c2 = -1/np.sqrt(2)
wfxn.add(d1,c1)
wfxn.add(d2,c2)

wfxn

|aa00000000000000000000000000000000000000000000000000000000000000> * (0.70710678,0.00000000)
|00aa000000000000000000000000000000000000000000000000000000000000> * (-0.70710678,0.00000000)

In [64]:
def one_rdm(n_spatial,wavefunction):
    pmat_a = np.zeros((n_spatial, n_spatial))
    pmat_b = np.zeros((n_spatial, n_spatial))
    
    for p in range(n_spatial):
        for q in range(n_spatial):
            gamma_a = forte2.SparseOperator()
            gamma_a.add(f"[{p}a+{q}a-]")
            wfxn2 = gamma_a @ wavefunction
            pmat_a[p,q] = wavefunction.overlap(wfxn2).real

            gamma_b = forte2.SparseOperator()
            gamma_b.add(f"[{p}b+{q}b-]")
            wfxn3 = gamma_b @ wavefunction
            pmat_b[p,q] = wavefunction.overlap(wfxn3).real

    return pmat_a, pmat_b

In [65]:
pa, pb = one_rdm(4, wfxn)
print(pa)


[[0.5 0.  0.  0. ]
 [0.  0.5 0.  0. ]
 [0.  0.  0.5 0. ]
 [0.  0.  0.  0.5]]


#### Code the 2-RDM using forte2

In [66]:
def two_rdm(n_spatial,wavefunction):
    pmat_aa = np.zeros((n_spatial, n_spatial, n_spatial, n_spatial))
    pmat_ab = np.zeros((n_spatial, n_spatial, n_spatial, n_spatial))
    pmat_bb = np.zeros((n_spatial, n_spatial, n_spatial, n_spatial))
    #index_list = np.linspace(0,n_spatial,n_spatial+1)
    # Case one: both electrons are alpha electrons:
    for p in range(n_spatial):
        for q in range(n_spatial):
            if p>q:
                gamma_left=forte2.SparseOperator()
                gamma_left.add(f"[{p}a-{q}a-]",1.0)
                wfxn_left= gamma_left @ wavefunction
                for r in range(n_spatial):
                    for s in range(n_spatial):
                        if r>s:
                            gamma_right=forte2.SparseOperator()
                            gamma_right.add(f"[{r}a-{s}a-]",1.0)
                            wfxn_right= gamma_right @ wavefunction
                            pmat_aa[p,q,r,s] = wfxn_left.overlap(wfxn_right).real
                            pmat_aa[q,p,s,r] = pmat_aa[p,q,r,s]
                            pmat_aa[q,p,r,s] = -pmat_aa[p,q,r,s]
                            pmat_aa[p,q,s,r] = -pmat_aa[p,q,r,s]
    # Case 2: both electrons are beta electrons:
    for p in range(n_spatial):
        for q in range(n_spatial):
            if p>q:
                gamma_left=forte2.SparseOperator()
                gamma_left.add(f"[{p}b-{q}b-]",1.0)
                wfxn_left= gamma_left @ wavefunction
                for r in range(n_spatial):
                    for s in range(n_spatial):
                        if r>s:
                            gamma_right=forte2.SparseOperator()
                            gamma_right.add(f"[{r}b-{s}b-]",1.0)
                            wfxn_right= gamma_right @ wavefunction
                            pmat_bb[p,q,r,s] = wfxn_left.overlap(wfxn_right).real
                            pmat_bb[q,p,s,r] = pmat_bb[p,q,r,s]
                            pmat_bb[q,p,r,s] = -pmat_bb[p,q,r,s]
                            pmat_bb[p,q,s,r] = -pmat_bb[p,q,r,s]
    #Case 3: Abba
    for p in range(n_spatial):
        for q in range(n_spatial):
            gamma_right=forte2.SparseOperator()
            gamma_right.add(f"[{p}b-{q}a-]",1.0)
            wfxn_right = gamma_right @ wavefunction
            for r in range(n_spatial):
                for s in range(n_spatial):
                    gamma_left = forte2.SparseOperator()
                    gamma_left.add(f"[{r}b-{s}a-]",1.0)
                    wfxn_left = gamma_left @ wavefunction
                    pmat_ab[p,q,r,s] = wfxn_left.overlap(wfxn_right).real
    return pmat_aa, pmat_bb, pmat_ab

In [67]:
wfxn2 = forte2.SparseState()
d1 = forte2.Determinant("abba")
d2 = forte2.Determinant("baab")
c1 = 1/np.sqrt(2)
c2 = -1/np.sqrt(2)
wfxn2.add(d1,c1)
wfxn2.add(d2,c2)

p_aa, p_bb, p_ab = two_rdm(4, wfxn2)
p_a, p_b = one_rdm(4, wfxn2)

mc_mat_aa = np.zeros((4,4,4,4))
mc_mat_ab = np.zeros((4,4,4,4))
mc_mat_bb = np.zeros((4,4,4,4))

for p, q, r, s in itertools.product(range(4), repeat=4):
    mc_mat_aa[p,q,r,s] = p_aa[p,q,r,s] - (p_a[p,r] * p_a[q,s] - p_a[p,s] * p_a[q,r])
    mc_mat_ab[p,q,r,s] = p_ab[p,q,r,s] - (p_a[p,r] * p_b[q,s])
    mc_mat_bb[p,q,r,s] = p_bb[p,q,r,s] - (p_b[p,r] * p_b[q,s] - p_b[p,s] * p_b[q,r])

frob_aa =0
frob_ab =0
frob_bb =0

for i, j, k, l in itertools.product(range(4), repeat=4):
    frob_aa += mc_mat_aa[i,j,k,l]**2
    frob_ab += mc_mat_ab[i,j,k,l]**2
    frob_bb += mc_mat_bb[i,j,k,l]**2

C = (frob_aa + 4*frob_ab + frob_bb)*0.25
print("Total Correlation")
print(C)
#print(trace)


Total Correlation
1.7499999999999996


In [68]:
j = itertools.product(range(4), repeat=4)
print(list(j))

[(0, 0, 0, 0), (0, 0, 0, 1), (0, 0, 0, 2), (0, 0, 0, 3), (0, 0, 1, 0), (0, 0, 1, 1), (0, 0, 1, 2), (0, 0, 1, 3), (0, 0, 2, 0), (0, 0, 2, 1), (0, 0, 2, 2), (0, 0, 2, 3), (0, 0, 3, 0), (0, 0, 3, 1), (0, 0, 3, 2), (0, 0, 3, 3), (0, 1, 0, 0), (0, 1, 0, 1), (0, 1, 0, 2), (0, 1, 0, 3), (0, 1, 1, 0), (0, 1, 1, 1), (0, 1, 1, 2), (0, 1, 1, 3), (0, 1, 2, 0), (0, 1, 2, 1), (0, 1, 2, 2), (0, 1, 2, 3), (0, 1, 3, 0), (0, 1, 3, 1), (0, 1, 3, 2), (0, 1, 3, 3), (0, 2, 0, 0), (0, 2, 0, 1), (0, 2, 0, 2), (0, 2, 0, 3), (0, 2, 1, 0), (0, 2, 1, 1), (0, 2, 1, 2), (0, 2, 1, 3), (0, 2, 2, 0), (0, 2, 2, 1), (0, 2, 2, 2), (0, 2, 2, 3), (0, 2, 3, 0), (0, 2, 3, 1), (0, 2, 3, 2), (0, 2, 3, 3), (0, 3, 0, 0), (0, 3, 0, 1), (0, 3, 0, 2), (0, 3, 0, 3), (0, 3, 1, 0), (0, 3, 1, 1), (0, 3, 1, 2), (0, 3, 1, 3), (0, 3, 2, 0), (0, 3, 2, 1), (0, 3, 2, 2), (0, 3, 2, 3), (0, 3, 3, 0), (0, 3, 3, 1), (0, 3, 3, 2), (0, 3, 3, 3), (1, 0, 0, 0), (1, 0, 0, 1), (1, 0, 0, 2), (1, 0, 0, 3), (1, 0, 1, 0), (1, 0, 1, 1), (1, 0, 1, 2), (1, 0

### -------------------------------------- Mutual Correlation --------------------------------------

In [97]:
def mutual_correlation(n_spatial,determinants,coeffs):
    wavefunction = forte2.SparseState()
    for d in determinants:
        c = coeffs[determinants.index(d)]
        wavefunction.add(d,c)
   
    paa, pbb, pab = two_rdm(n_spatial,wavefunction)
    pa, pb = one_rdm(n_spatial,wavefunction)

    #Calculate Total Correlation   
    mc_mat_aa = paa - (np.einsum('pr,qs->pqrs',pa,pa) - np.einsum('ps,qr->pqrs',pa,pa))
    mc_mat_ab = pab - (np.einsum('pr,qs->pqrs',pa,pb))
    mc_mat_bb = pbb - (np.einsum('pr,qs->pqrs',pb,pb) - np.einsum('ps,qr->pqrs',pb,pb))

    frob_aa = np.sum(mc_mat_aa**2)
    frob_ab = np.sum(mc_mat_ab**2)
    frob_bb = np.sum(mc_mat_bb**2)
    
    C = (frob_aa + 4*frob_ab + frob_bb)*0.25
    #Build the mutual correlation matrix
    lambda_2_mat = (mc_mat_aa**2 + 
                    mc_mat_ab**2 + 
                    np.transpose(mc_mat_ab, (1,0,2,3))**2 + 
                    np.transpose(mc_mat_ab, (0,1,3,2))**2 + 
                    np.transpose(mc_mat_ab, (1,0,3,2))**2 + 
                    mc_mat_bb**2
                    )   
    
    mc_mat = (np.einsum('pqqq->pq', lambda_2_mat) 
        + 0.5*np.einsum('ppqq->pq', lambda_2_mat) 
        + 0.5*np.einsum('pqpq->pq', lambda_2_mat) 
        + 0.5*np.einsum('pqqp->pq', lambda_2_mat) 
        + np.einsum('ppqp->pq', lambda_2_mat)
        )
    
    print(np.einsum('pqpq->pq', mc_mat_ab))
    print(np.einsum('pqqp->pq', mc_mat_ab))

    for p, q in itertools.product(range(n_spatial), repeat=2):       
        if q>p:
            print(f"mc_mat[{p},{q}] = {mc_mat[p,q]}")   
    
    return C, mc_mat_aa, mc_mat_ab, mc_mat_bb   


In [98]:
#wfxn = forte2.SparseState()
d1 = forte2.Determinant("ab00")
d2 = forte2.Determinant("a0b0")
d3 = forte2.Determinant("0ab0")
dets = [d1,d2,d3]
c1 = np.random.rand()
c2 = np.random.rand()
c3 = np.random.rand()
coeff = [c1,c2,c3]
#wfxn.add(d1,c1)
#wfxn.add(d2,c2)

#FROM MC PAPER: EXPECTED VALUE OF FROB NORM: 1.750
#EXPECTED VALUE OF TRACE: 2.0

MC = mutual_correlation(4,dets,coeff)
#plt.imshow(MC[0], cmap='PiYG')
print(MC[0])
print(dets)

[[ 0.         -0.25154664 -0.23509939  0.        ]
 [ 0.50051261 -0.23309944 -0.21785835  0.        ]
 [ 0.00206542  0.46572142  0.          0.        ]
 [ 0.          0.          0.          0.        ]]
[[ 0.          0.          0.          0.        ]
 [ 0.         -0.23309944  0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]]
mc_mat[0,1] = 0.3686058830990721
mc_mat[0,2] = 0.055275988261978694
mc_mat[0,3] = 0.0
mc_mat[1,2] = 0.31914249216986024
mc_mat[1,3] = 0.0
mc_mat[2,3] = 0.0
1.268497143292781
[|ab00000000000000000000000000000000000000000000000000000000000000>, |a0b0000000000000000000000000000000000000000000000000000000000000>, |0ab0000000000000000000000000000000000000000000000000000000000000>]


In [71]:
#FROM MC PAPER: EXPECTED VALUE OF FROB NORM: 1.750
#EXPECTED VALUE OF TRACE: 2.0

d1 = forte2.Determinant("0220")
d2 = forte2.Determinant("2002")
c1 = 1/np.sqrt(2)
c2 = -1/np.sqrt(2)
dets = [d1,d2]
coeff = [c1,c2]


MC = mutual_correlation(4,dets,coeff)
print(MC[0])
print(dets)

mc_mat[0,1] = 0.24999999999999978
mc_mat[0,2] = 0.24999999999999978
mc_mat[0,3] = 0.25
mc_mat[1,2] = 0.25
mc_mat[1,3] = 0.24999999999999978
mc_mat[2,3] = 0.24999999999999978
1.7499999999999991
[|0220000000000000000000000000000000000000000000000000000000000000>, |2002000000000000000000000000000000000000000000000000000000000000>]


#### Implementing Mutual Correlation for the Hubbard Model

In [72]:
def hubbard_hamiltonian_v2(nsites,nalpha,nbeta,U,T):
    dets = forte2.hilbert_space(nsites,nalpha,nbeta)
    ndets = len(dets)
    H = np.zeros((ndets,ndets))
    dets_i = {dets[i]: i for i in range(ndets)}

    #diagonal: if site is doubly occupied, H[m,m]=U
    for j in range(0,len(dets)):
        for i in range(0,nsites):
            var = str(dets[j].str(nsites)[i+1])
            if (str(var) == '2'):
                H[j,j] += U
    
    #off-diagonal: if annop(site) and creop(site+1) both are allowed, H[mn] = t 
    alpha_ops = []
    beta_ops = []
    for k in range(nsites-1):
        alpha = forte2.SparseOperator()
        alpha.add(f"[{k+1}a+{k}a-]")
        alpha_ops.append(alpha)
        beta = forte2.SparseOperator()
        beta.add(f"[{k+1}b+{k}b-]")
        beta_ops.append(beta)
    for j, det in enumerate(dets): #loops through both j and dets[j]
        wfxn = forte2.SparseState()
        wfxn.add(det,1.0)
        for op in (alpha_ops + beta_ops):
            f = op @ wfxn
            if f:
                for new_det, coeff in f.items():
                    i = dets_i[new_det]
                    H[i,j] += -T 
    H = H + H.T - np.diag(np.diag(H)) #copies symmetry into lower triangular without re-looping
    return H, dets

In [73]:
""" H, dets = hubbard_hamiltonian_v2(8,4,4,8,1)

evals, evecs = np.linalg.eigh(H)
#print(evecs[0,:])
coeffs = evecs[:,0]

MC = mutual_correlation(8,dets,coeffs)
print(MC[0]) """

' H, dets = hubbard_hamiltonian_v2(8,4,4,8,1)\n\nevals, evecs = np.linalg.eigh(H)\n#print(evecs[0,:])\ncoeffs = evecs[:,0]\n\nMC = mutual_correlation(8,dets,coeffs)\nprint(MC[0]) '

In [74]:
""" H, dets = hubbard_hamiltonian_v2(8,4,4,1,1)

evals, evecs = np.linalg.eigh(H)
#print(evecs[0,:])
coeffs = evecs[:,0]

MC = mutual_correlation(8,dets,coeffs)
print(MC[0]) """

' H, dets = hubbard_hamiltonian_v2(8,4,4,1,1)\n\nevals, evecs = np.linalg.eigh(H)\n#print(evecs[0,:])\ncoeffs = evecs[:,0]\n\nMC = mutual_correlation(8,dets,coeffs)\nprint(MC[0]) '

In [75]:
""" H, dets = hubbard_hamiltonian_v2(8,4,4,8,1)

evals, evecs = np.linalg.eigh(H)
#print(evecs[0,:])
coeffs = evecs[:,1]

MC = mutual_correlation(8,dets,coeffs)
print(MC[0]) """

' H, dets = hubbard_hamiltonian_v2(8,4,4,8,1)\n\nevals, evecs = np.linalg.eigh(H)\n#print(evecs[0,:])\ncoeffs = evecs[:,1]\n\nMC = mutual_correlation(8,dets,coeffs)\nprint(MC[0]) '

In [76]:
""" H, dets = hubbard_hamiltonian_v2(8,4,4,8,1)

evals, evecs = np.linalg.eigh(H)
#print(evecs[0,:])
coeffs = evecs[:,0]

MC = mutual_correlation(8,dets,coeffs) """

' H, dets = hubbard_hamiltonian_v2(8,4,4,8,1)\n\nevals, evecs = np.linalg.eigh(H)\n#print(evecs[0,:])\ncoeffs = evecs[:,0]\n\nMC = mutual_correlation(8,dets,coeffs) '

#### Mutual Correlation Energy **old functions!

In [77]:
def subspaces(*orbital_subspaces, rdm_orbitals=None):
    frag_orbs = []
    seen = set()

    for subspace in orbital_subspaces:
        for orb in subspace:
            if orb in seen:
                raise ValueError(f"Orbital {orb} appears in more than one subspace")
            seen.add(orb)
            frag_orbs.append(orb)

    local_pos = {orb: i for i, orb in enumerate(frag_orbs)}
    local_subspaces = [
        [local_pos[orb] for orb in subspace]
        for subspace in orbital_subspaces
    ]

    if rdm_orbitals is None:
        global_subspaces = [list(subspace) for subspace in orbital_subspaces]
    else:
        rdm_pos = {orb: i for i, orb in enumerate(rdm_orbitals)}
        missing = sorted(set(frag_orbs) - set(rdm_pos))
        if missing:
            raise ValueError(
                f"Fragment orbitals are not present in the RDM orbital space: {missing}"
            )

        global_subspaces = [
            [rdm_pos[orb] for orb in subspace]
            for subspace in orbital_subspaces
        ]

    return frag_orbs, global_subspaces, local_subspaces


In [78]:
import forte2

xyz = """
H 0.000 0.000 0.000
H 0.000 0.000 1.740
H 0.000 0.000 3.480
"""

system = forte2.System(xyz=xyz, basis_set="cc-pVDZ", auxiliary_basis_set="cc-pVTZ-JKFIT")

rhf = forte2.ROHF(charge=0,ms=0.5)(system)
rhf.run()

norb = rhf.nmo
orbitals = sorted(list(set(range(norb))))
print(orbitals)

ci=forte2.CI(forte2.State(system=system, multiplicity=2,ms=0.5),active_orbitals=[0,1,2,3,4,5,6,7])(rhf)
ci.run()

frag_orbs, A_glob, A_loc = subspaces([0,1,2,3,4,5,6,7])
ints = forte2.jkbuilder.RestrictedMOIntegrals(
    system=ci.system,
    C=ci.C[0],
    orbitals=frag_orbs,
    core_orbitals=())

Ecore = ints.E

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   H   0.00000000   0.00000000   0.00000000
   H   0.00000000   0.00000000   3.28812346
   H   0.00000000   0.00000000   6.57624691
Parsed 3 atoms with basis set of 15 functions.
  Max eigenvalue: 2.532e+00
  Min eigenvalue: 1.284e-01
  Condition number: 1.972e+01
  Inverse condition number: 5.071e-02
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 15
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 1.284e-01
Number of electrons: 3
Number of alpha electrons: 2
Number of beta electrons: 1
Ms: 0.5
Total charge: 0
Number of basis functions: 15
Number of orthogonalized basis functions: 15
Number of auxiliary basis functions: 90
Energy convergence criterion: 1.000000e-09
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> ROHF SCF ROUTINE <==
Memory requirements: 0.00 GB (doubled due to storing B_nPm)
Number of system basis fun

In [79]:
def onefrag_correlation_energy(ci, A_orbs, core_orbitals=()):
    system = ci.system
    C = ci.C[0]
    ci_solver = ci.sub_solvers[0]

    a1, b1 = ci_solver.make_sd_1rdm(0)
    aa_pair, ab2, bb_pair = ci_solver.make_sd_2rdm(0)

    aa2 = forte2.cpp_helpers.packed_tensor4_to_tensor4(aa_pair)
    bb2 = forte2.cpp_helpers.packed_tensor4_to_tensor4(bb_pair)

    frag_orbs, global_subs, local_subs = subspaces(A_orbs, rdm_orbitals=ci.mo_space.active_indices)
    A_glob = np.array(global_subs[0], dtype=int)
    A_loc = np.array(local_subs[0], dtype=int)

    ints = forte2.jkbuilder.RestrictedMOIntegrals(
        system=system,
        C=C,
        orbitals=frag_orbs,
        core_orbitals=list(core_orbitals),
    )
    H = ints.H
    V = ints.V

    ea_1body = (
        np.einsum("pq,pq->", H[np.ix_(A_loc, A_loc)], a1[np.ix_(A_glob, A_glob)])
        + np.einsum("pq,pq->", H[np.ix_(A_loc, A_loc)], b1[np.ix_(A_glob, A_glob)])
    )

    ea_2body = 0.5 * (
        np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, A_loc, A_loc, A_loc)], aa2[np.ix_(A_glob, A_glob, A_glob, A_glob)])
        + 2.0 * np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, A_loc, A_loc, A_loc)], ab2[np.ix_(A_glob, A_glob, A_glob, A_glob)])
        + np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, A_loc, A_loc, A_loc)], bb2[np.ix_(A_glob, A_glob, A_glob, A_glob)])
    )

    return ea_1body + ea_2body

In [80]:
def twofrag_correlation_energy(ci, A_orbs, B_orbs, core_orbitals=()):
    overlap = set(A_orbs) & set(B_orbs)
    if overlap:
        raise ValueError(
            f"A_orbs and B_orbs must be disjoint; shared orbitals found: {sorted(overlap)}"
        )

    system = ci.system
    C = ci.C[0]
    ci_solver = ci.sub_solvers[0]

    a1, b1 = ci_solver.make_sd_1rdm(0)
    aa_pair, ab2, bb_pair = ci_solver.make_sd_2rdm(0)

    aa2 = forte2.cpp_helpers.packed_tensor4_to_tensor4(aa_pair)
    bb2 = forte2.cpp_helpers.packed_tensor4_to_tensor4(bb_pair)

    frag_orbs, global_subs, local_subs = subspaces(A_orbs, B_orbs, rdm_orbitals=ci.mo_space.active_indices)
    A_glob, B_glob = [np.array(x, dtype=int) for x in global_subs]
    A_loc, B_loc = [np.array(x, dtype=int) for x in local_subs]

    ints = forte2.jkbuilder.RestrictedMOIntegrals(
        system=system,
        C=C,
        orbitals=frag_orbs,
        core_orbitals=list(core_orbitals),
    )
    H = ints.H
    V = ints.V

    # one body correlation
    e1_a = 2 * np.einsum("pq,pq->",H[np.ix_(A_loc, B_loc)],a1[np.ix_(A_glob, B_glob)])
    e1_b = 2 * np.einsum("pq,pq->",H[np.ix_(A_loc, B_loc)],b1[np.ix_(A_glob, B_glob)])
    Total_1bc = e1_a + e1_b

    # two body correlation
    Total_2bc = 0.5 * (
        4 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, A_loc, A_loc, B_loc)],aa2[np.ix_(A_glob, A_glob, A_glob, B_glob)],optimize=True)
        + 2 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, A_loc, B_loc, B_loc)],aa2[np.ix_(A_glob, A_glob, B_glob, B_glob)],optimize=True)
        + 2 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, B_loc, A_loc, B_loc)],aa2[np.ix_(A_glob, B_glob, A_glob, B_glob)],optimize=True)
        + 2 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, B_loc, B_loc, A_loc)],aa2[np.ix_(A_glob, B_glob, B_glob, A_glob)],optimize=True)
        + 4 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, B_loc, B_loc, B_loc)],aa2[np.ix_(A_glob, B_glob, B_glob, B_glob)],optimize=True)
        
        + 8 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, A_loc, A_loc, B_loc)],ab2[np.ix_(A_glob, A_glob, A_glob, B_glob)],optimize=True)
        + 4 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, A_loc, B_loc, B_loc)],ab2[np.ix_(A_glob, A_glob, B_glob, B_glob)],optimize=True)
        + 4 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, B_loc, A_loc, B_loc)],ab2[np.ix_(A_glob, B_glob, A_glob, B_glob)],optimize=True)
        + 4 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, B_loc, B_loc, A_loc)],ab2[np.ix_(A_glob, B_glob, B_glob, A_glob)],optimize=True)
        + 8 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, B_loc, B_loc, B_loc)],ab2[np.ix_(A_glob, B_glob, B_glob, B_glob)],optimize=True)
        
        + 4 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, A_loc, A_loc, B_loc)],bb2[np.ix_(A_glob, A_glob, A_glob, B_glob)],optimize=True)
        + 2 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, A_loc, B_loc, B_loc)],bb2[np.ix_(A_glob, A_glob, B_glob, B_glob)],optimize=True)
        + 2 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, B_loc, A_loc, B_loc)],bb2[np.ix_(A_glob, B_glob, A_glob, B_glob)],optimize=True)
        + 2 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, B_loc, B_loc, A_loc)],bb2[np.ix_(A_glob, B_glob, B_glob, A_glob)],optimize=True)
        + 4 * np.einsum("pqrs,pqrs->",V[np.ix_(A_loc, B_loc, B_loc, B_loc)],bb2[np.ix_(A_glob, B_glob, B_glob, B_glob)],optimize=True)
    )

    return Total_1bc + Total_2bc


In [81]:
#test: Etotal = E_core + E_A + E_B + E_AB for two subspaces A, B

E_A = onefrag_correlation_energy(ci=ci, A_orbs=[0,1,2,3])
E_B = onefrag_correlation_energy(ci=ci, A_orbs=[4,5,6,7])

E_AB = twofrag_correlation_energy(ci=ci,A_orbs=[0,1,2,3],B_orbs=[4,5,6,7])

print(ci.E-(Ecore+E_A+E_B))
print(E_AB)

[-0.02184268]
-0.015735909142317914


In [82]:
def threefrag_correlation_energy(ci, A_orbs, B_orbs, C_orbs, core_orbitals=()):
    overlap = ((set(A_orbs) & set(B_orbs))
            | (set(A_orbs) & set(C_orbs))
            | (set(B_orbs) & set(C_orbs)))

    if overlap:
        raise ValueError(f"Orbital fragments must be disjoint; shared orbitals found: {sorted(overlap)}")

    system = ci.system
    C = ci.C[0]
    ci_solver = ci.sub_solvers[0]

    #a1, b1 = ci_solver.make_sd_1rdm(0)
    aa_pair, ab2, bb_pair = ci_solver.make_sd_2rdm(0)

    aa2 = forte2.cpp_helpers.packed_tensor4_to_tensor4(aa_pair)
    bb2 = forte2.cpp_helpers.packed_tensor4_to_tensor4(bb_pair)

    frag_orbs, global_subs, local_subs = subspaces(A_orbs, B_orbs, C_orbs, rdm_orbitals=ci.mo_space.active_indices,)
    A_glob, B_glob, C_glob = [np.array(x, dtype=int) for x in global_subs]
    A_loc, B_loc, C_loc = [np.array(x, dtype=int) for x in local_subs]

    ints = forte2.jkbuilder.RestrictedMOIntegrals(
        system=system,
        C=C,
        orbitals=frag_orbs,
        core_orbitals=list(core_orbitals),
    )
    V = ints.V

    #print(np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, A_loc, B_loc, C_loc)], aa2[np.ix_(A_glob, A_glob, B_glob, C_glob)], optimize=True))
    #print(np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, A_loc, C_loc)], aa2[np.ix_(A_glob, B_glob, A_glob, C_glob)], optimize=True))
    #print(np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, C_loc, A_loc)], aa2[np.ix_(A_glob, B_glob, C_glob, A_glob)], optimize=True))
          
    #print(np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, B_loc, A_loc, C_loc)], aa2[np.ix_(B_glob, B_glob, A_glob, C_glob)], optimize=True))
    #print(np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, B_loc, C_loc)], aa2[np.ix_(B_glob, A_glob, B_glob, C_glob)], optimize=True))
    #print(np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, C_loc, B_loc)], aa2[np.ix_(B_glob, A_glob, C_glob, B_glob)], optimize=True))

    #print(np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, C_loc, A_loc, B_loc)], aa2[np.ix_(C_glob, C_glob, A_glob, B_glob)], optimize=True))
    #print(np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, C_loc, B_loc)], aa2[np.ix_(C_glob, A_glob, C_glob, B_glob)], optimize=True))
    print(np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, B_loc, A_loc, C_loc)], aa2[np.ix_(C_glob, B_glob, A_glob, C_glob)], optimize=True))

    Correlation = 0.5*(
    4*(np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, A_loc, B_loc, C_loc)], aa2[np.ix_(A_glob, A_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, A_loc, C_loc)], aa2[np.ix_(A_glob, B_glob, A_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, C_loc, A_loc)], aa2[np.ix_(A_glob, B_glob, C_glob, A_glob)], optimize=True)
    
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, B_loc, A_loc, C_loc)], aa2[np.ix_(B_glob, B_glob, A_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, B_loc, C_loc)], aa2[np.ix_(B_glob, A_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, C_loc, B_loc)], aa2[np.ix_(B_glob, A_glob, C_glob, B_glob)], optimize=True)

    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, C_loc, A_loc, B_loc)], aa2[np.ix_(C_glob, C_glob, A_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, C_loc, B_loc)], aa2[np.ix_(C_glob, A_glob, C_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, B_loc, C_loc)], aa2[np.ix_(C_glob, A_glob, B_glob, C_glob)], optimize=True))

    +8*(np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, A_loc, B_loc, C_loc)], ab2[np.ix_(A_glob, A_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, A_loc, C_loc)], ab2[np.ix_(A_glob, B_glob, A_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, C_loc, A_loc)], ab2[np.ix_(A_glob, B_glob, C_glob, A_glob)], optimize=True)
    
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, B_loc, A_loc, C_loc)], ab2[np.ix_(B_glob, B_glob, A_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, B_loc, C_loc)], ab2[np.ix_(B_glob, A_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, C_loc, B_loc)], ab2[np.ix_(B_glob, A_glob, C_glob, B_glob)], optimize=True)

    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, C_loc, A_loc, B_loc)], ab2[np.ix_(C_glob, C_glob, A_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, C_loc, B_loc)], ab2[np.ix_(C_glob, A_glob, C_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, B_loc, C_loc)], ab2[np.ix_(C_glob, A_glob, B_glob, C_glob)], optimize=True))
    
    +4*(np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, A_loc, B_loc, C_loc)], bb2[np.ix_(A_glob, A_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, A_loc, C_loc)], bb2[np.ix_(A_glob, B_glob, A_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, C_loc, A_loc)], bb2[np.ix_(A_glob, B_glob, C_glob, A_glob)], optimize=True)
    
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, B_loc, A_loc, C_loc)], bb2[np.ix_(B_glob, B_glob, A_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, B_loc, C_loc)], bb2[np.ix_(B_glob, A_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, C_loc, B_loc)], bb2[np.ix_(B_glob, A_glob, C_glob, B_glob)], optimize=True)

    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, C_loc, A_loc, B_loc)], bb2[np.ix_(C_glob, C_glob, A_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, C_loc, B_loc)], bb2[np.ix_(C_glob, A_glob, C_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, B_loc, C_loc)], bb2[np.ix_(C_glob, A_glob, B_glob, C_glob)], optimize=True))
    )
    return Correlation

In [83]:
E_A = onefrag_correlation_energy(ci=ci, A_orbs=[0,1,2])
E_B = onefrag_correlation_energy(ci=ci, A_orbs=[3,4,5])
E_C = onefrag_correlation_energy(ci=ci, A_orbs=[6,7])

E_AB = twofrag_correlation_energy(ci=ci,A_orbs=[0,1,2],B_orbs=[3,4,5])
E_BC = twofrag_correlation_energy(ci=ci,A_orbs=[3,4,5],B_orbs=[6,7])
E_AC = twofrag_correlation_energy(ci=ci,A_orbs=[0,1,2],B_orbs=[6,7])

E_ABC = threefrag_correlation_energy(ci=ci, A_orbs=[0,1,2], B_orbs=[3,4,5], C_orbs=[6,7])

CI_ABC = (ci.E-(Ecore+E_A+E_B+E_C +E_AB+E_BC+E_AC))
print(CI_ABC)
print(E_ABC)

print(abs(CI_ABC-E_ABC))

-1.2271014517360049e-06
[-0.00312987]
-0.0008212016302238238
[0.00230866]


In [84]:
def fourfrag_correlation_energy(ci, A_orbs, B_orbs, C_orbs, D_orbs, core_orbitals=()):
    fragments = [A_orbs, B_orbs, C_orbs, D_orbs]

    overlap = set()
    for i in range(len(fragments)):
        for j in range(i + 1, len(fragments)):
            overlap |= set(fragments[i]) & set(fragments[j])

    if overlap:
        raise ValueError(
            f"Orbital fragments must be disjoint; shared orbitals found: {sorted(overlap)}"
        )

    system = ci.system
    mo_coeff = ci.C[0]
    ci_solver = ci.sub_solvers[0]

    aa_pair, ab2, bb_pair = ci_solver.make_sd_2rdm(0)

    aa2 = forte2.cpp_helpers.packed_tensor4_to_tensor4(aa_pair)
    bb2 = forte2.cpp_helpers.packed_tensor4_to_tensor4(bb_pair)

    frag_orbs, global_subs, local_subs = subspaces(A_orbs, B_orbs, C_orbs, D_orbs, rdm_orbitals=ci.mo_space.active_indices)

    A_glob, B_glob, C_glob, D_glob = [np.array(x, dtype=int) for x in global_subs]
    A_loc, B_loc, C_loc, D_loc = [np.array(x, dtype=int) for x in local_subs]

    ints = forte2.jkbuilder.RestrictedMOIntegrals(
        system=system,
        C=mo_coeff,
        orbitals=frag_orbs,
        core_orbitals=list(core_orbitals),
    )
    V = ints.V
    corr = (
    np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, C_loc, D_loc)], aa2[np.ix_(A_glob, B_glob, C_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, D_loc, C_loc)], aa2[np.ix_(A_glob, B_glob, D_glob, C_glob)], optimize=True)

    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, C_loc, B_loc, D_loc)], aa2[np.ix_(A_glob, C_glob, B_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, C_loc, D_loc, B_loc)], aa2[np.ix_(A_glob, C_glob, D_glob, B_glob)], optimize=True)

    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, D_loc, B_loc, C_loc)], aa2[np.ix_(A_glob, D_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, D_loc, C_loc, B_loc)], aa2[np.ix_(A_glob, D_glob, C_glob, B_glob)], optimize=True)

    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, C_loc, D_loc)], aa2[np.ix_(B_glob, A_glob, C_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, D_loc, C_loc)], aa2[np.ix_(B_glob, A_glob, D_glob, C_glob)], optimize=True)

    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, B_loc, D_loc)], aa2[np.ix_(C_glob, A_glob, B_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, D_loc, B_loc)], aa2[np.ix_(C_glob, A_glob, D_glob, B_glob)], optimize=True)
    
    +np.einsum("pqrs,pqrs->", V[np.ix_(D_loc, A_loc, B_loc, C_loc)], aa2[np.ix_(D_glob, A_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(D_loc, A_loc, C_loc, B_loc)], aa2[np.ix_(D_glob, A_glob, C_glob, B_glob)], optimize=True)


    +(2)*(np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, C_loc, D_loc)], ab2[np.ix_(A_glob, B_glob, C_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, D_loc, C_loc)], ab2[np.ix_(A_glob, B_glob, D_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, C_loc, B_loc, D_loc)], ab2[np.ix_(A_glob, C_glob, B_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, C_loc, D_loc, B_loc)], ab2[np.ix_(A_glob, C_glob, D_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, D_loc, B_loc, C_loc)], ab2[np.ix_(A_glob, D_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, D_loc, C_loc, B_loc)], ab2[np.ix_(A_glob, D_glob, C_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, C_loc, D_loc)], ab2[np.ix_(B_glob, A_glob, C_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, D_loc, C_loc)], ab2[np.ix_(B_glob, A_glob, D_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, B_loc, D_loc)], ab2[np.ix_(C_glob, A_glob, B_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, D_loc, B_loc)], ab2[np.ix_(C_glob, A_glob, D_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(D_loc, A_loc, B_loc, C_loc)], ab2[np.ix_(D_glob, A_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(D_loc, A_loc, C_loc, B_loc)], ab2[np.ix_(D_glob, A_glob, C_glob, B_glob)], optimize=True))


    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, C_loc, D_loc)], bb2[np.ix_(A_glob, B_glob, C_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, B_loc, D_loc, C_loc)], bb2[np.ix_(A_glob, B_glob, D_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, C_loc, B_loc, D_loc)], bb2[np.ix_(A_glob, C_glob, B_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, C_loc, D_loc, B_loc)], bb2[np.ix_(A_glob, C_glob, D_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, D_loc, B_loc, C_loc)], bb2[np.ix_(A_glob, D_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(A_loc, D_loc, C_loc, B_loc)], bb2[np.ix_(A_glob, D_glob, C_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, C_loc, D_loc)], bb2[np.ix_(B_glob, A_glob, C_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(B_loc, A_loc, D_loc, C_loc)], bb2[np.ix_(B_glob, A_glob, D_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, B_loc, D_loc)], bb2[np.ix_(C_glob, A_glob, B_glob, D_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(C_loc, A_loc, D_loc, B_loc)], bb2[np.ix_(C_glob, A_glob, D_glob, B_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(D_loc, A_loc, B_loc, C_loc)], bb2[np.ix_(D_glob, A_glob, B_glob, C_glob)], optimize=True)
    +np.einsum("pqrs,pqrs->", V[np.ix_(D_loc, A_loc, C_loc, B_loc)], bb2[np.ix_(D_glob, A_glob, C_glob, B_glob)], optimize=True))

    return corr


In [85]:

E_A = onefrag_correlation_energy(ci=ci, A_orbs=[0,1])
E_B = onefrag_correlation_energy(ci=ci, A_orbs=[2,3])
E_C = onefrag_correlation_energy(ci=ci, A_orbs=[4,5])
E_D = onefrag_correlation_energy(ci=ci, A_orbs=[6,7])

E_AB = twofrag_correlation_energy(ci=ci,A_orbs=[0,1],B_orbs=[2,3])
E_AC = twofrag_correlation_energy(ci=ci,A_orbs=[0,1],B_orbs=[4,5])
E_AD = twofrag_correlation_energy(ci=ci,A_orbs=[0,1],B_orbs=[6,7])
E_BC = twofrag_correlation_energy(ci=ci,A_orbs=[2,3],B_orbs=[4,5])
E_BD = twofrag_correlation_energy(ci=ci,A_orbs=[2,3],B_orbs=[6,7])
E_CD = twofrag_correlation_energy(ci=ci,A_orbs=[4,5],B_orbs=[6,7])

E_ABC = threefrag_correlation_energy(ci=ci, A_orbs=[0,1], B_orbs=[2,3], C_orbs=[4,5])
E_ABD = threefrag_correlation_energy(ci=ci, A_orbs=[0,1], B_orbs=[2,3], C_orbs=[6,7])
E_ACD = threefrag_correlation_energy(ci=ci, A_orbs=[0,1], B_orbs=[4,5], C_orbs=[6,7])
E_BCD = threefrag_correlation_energy(ci=ci, A_orbs=[2,3], B_orbs=[4,5], C_orbs=[6,7])

E_ABCD = fourfrag_correlation_energy(ci=ci, A_orbs=[0,1], B_orbs=[2,3], C_orbs=[4,5], D_orbs=[6,7])

ci_ABCD = (ci.E-(Ecore + E_A + E_B + E_C + E_D + E_AB + E_AC + E_AD +E_BC + E_BD + E_CD + E_ABC + E_ABD + E_ACD + E_BCD))

print(ci_ABCD)
print(E_ABCD)

print(abs(ci_ABCD-E_ABCD)) 

2.724877968830118e-05
3.538284012044187e-07
-4.707929491632336e-07
-2.8904898785511146e-07
[-0.06259846]
-0.0003138891293213707
[0.06228457]
